# Embedding Our Dataset
Embedding It's a neural network trained to capture meaning, so texts that mean similar things land on similar vectors.

## Loading the data

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
from ingest import load_faq_data

In [8]:
documents = load_faq_data()

## Generating embeddings

Each document is a Python dictionary with a question and an answer. We embed both together. That way a query can match against the question text and the answer text in our index.

In [9]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [10]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [11]:
texts[0]

"Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."

In [12]:
# Now we generate the embeddings.
# First we import tqdm so we can watch the progress
from tqdm.auto import tqdm

In [13]:
# Next we chunk the dataset into batches of 50 and encode each batch:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [14]:
# Give me the second vector in the vectors list.
vectors[1]

array([-1.09955193e-02, -1.10747442e-01, -2.53694188e-02,  1.97100881e-02,
       -8.37622806e-02, -1.55265510e-01,  6.08169772e-02,  3.26978578e-03,
       -1.84232727e-01,  3.61288637e-02, -2.03759335e-02,  8.37011542e-03,
        8.53649974e-02, -8.11423287e-02, -3.44118103e-02,  2.99428664e-02,
       -4.36164960e-02, -4.33529839e-02,  4.82881330e-02, -4.70028482e-02,
        1.30925924e-02, -3.58360298e-02,  3.54018472e-02, -2.82692239e-02,
        3.28568034e-02,  4.10358235e-02,  9.07346532e-02,  5.90873137e-02,
        1.11789508e-02,  1.03822583e-02, -8.85614473e-03,  2.35163383e-02,
        1.38390809e-02,  5.80496378e-02,  2.97546145e-02, -1.17032491e-02,
        1.00628614e-01, -1.04430988e-01, -9.16769952e-02,  1.54857021e-02,
       -7.07582105e-03,  3.44297215e-02,  1.17165819e-02, -2.63242293e-02,
        5.18438267e-03,  1.29875895e-02, -9.26766172e-03, -5.60835153e-02,
        1.92830488e-02, -9.90841817e-03, -5.16267233e-02, -5.70740327e-02,
       -8.25721622e-02, -

We turn them into a 2-dimensional array (matrix) where

rows are documents (vectors) <br>
columns are dimensions of the vectors

In [15]:
import numpy as np
X = np.array(vectors)

In [16]:
X

array([[-0.02670618, -0.12245757,  0.01594413, ..., -0.00230654,
        -0.11218394, -0.02365559],
       [-0.01099552, -0.11074744, -0.02536942, ...,  0.09022228,
        -0.02697371,  0.01975672],
       [-0.08896548, -0.06128178,  0.00775603, ...,  0.0405971 ,
         0.00479277, -0.02745943],
       ...,
       [-0.03652925,  0.01415426, -0.06838644, ...,  0.04316786,
         0.08105537, -0.02148626],
       [-0.13091588, -0.06990605, -0.0093188 , ..., -0.00044342,
        -0.0128573 ,  0.01426918],
       [-0.07984784,  0.01926981,  0.02544978, ..., -0.03368027,
        -0.01884026,  0.05837054]], shape=(1350, 384), dtype=float32)

In [17]:
X.shape

(1350, 384)

# Vector Search
In the previous lesson we embedded our FAQ dataset into a matrix X with 1208 document vectors. Here we see how vector search works under the hood.

We have a matrix X with all document embeddings. We take a query, compare it against every document, and keep the most similar ones

In [18]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [19]:
# we compute the dot product against all documents
scores = X.dot(v_query)


In [20]:
scores

array([ 0.48740587,  0.2099193 ,  0.762941  , ..., -0.08637964,
        0.03759784, -0.03037035], shape=(1350,), dtype=float32)

In [21]:
# The numpy.argmax() function returns the indices of the maximum values along a specified axis
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [22]:
idx

np.int64(2)

In [23]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

## Top 5 results

In [24]:
# np.argsort sorts from lowest to highest, so the last 5 are the top ones
top5 = np.argsort(scores)[-5:]

In [25]:
# They come out smallest-first, so we reverse them to get the highest first
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [26]:
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [27]:
# Let's read off the actual documents
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

## Vector Search with minsearch

## Creating the index

In [28]:
# We already have our documents and vectors from the previous section.
# Index them
from minsearch import VectorSearch

# The keyword_fields parameter works the same as in the text Index, so we can filter by course later.
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

## Searching

In [29]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [30]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [31]:
{"id": "74eb249bbf",
 "course": "llm-zoomcamp",
 "section": "General Course-Related Questions",
 "question": "I just discovered the course. Can I still join?",
 "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions."}

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

## Filtering by course
Like the text index, we can filter by keyword fields. This matters for user experience.

In [32]:
# Pass a filter_dict|
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# Vector Search with minsearch

It has a VectorSearch class for vector search. <br>

Both classes share the same API:<br>
fit to index data <br>
search to query <br>
filter_dict in search to filter by keyword <br>
It's the simplest way to get started with vector search.

## Creating the index
We already have our documents and vectors from the previous section. <br>
Index them:

In [33]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

## Searching
Under the hood it does the same thing we just did by hand. It computes the dot product between each vector (after filtering) and our query vector.

In [34]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [35]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [36]:
# Filtering by course
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# RAG with Vector Search
In module 1, we built a RAG pipeline with three steps. <br>
The search step used keyword search. Now we swap in vector search. Because RAG is modular, search is the only step we touch. Build prompt and the LLM call stay exactly as they were.

In [37]:
from rag_helper import RAGBase

In [38]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

## Using RAGBase
In module 1 we put all the RAG logic into a RAGBase helper class. It has search, build_prompt, and llm methods, so we only need to override search.

In [39]:
from ingest import build_index

In [40]:
# download and index the data
# we already dowloaded the data in the previous section, so we can skip that step here.
# documents = load_faq_data() 
index = build_index(documents)

In [41]:
# Then use the RAGBase class:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [42]:
# Ask it a question:
# This still uses keyword search. 
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

We already have: <br>
All the indexed documents documents <br>
The embeddings matrix X with all these documents <br>
The vector search engine vindex <br>

We can't pass vindex to RAG as-is. Text search takes the query string directly, but vector search needs the query as a vector first. So we subclass RAGBase and override search to encode the query before searching.

In [43]:
# The subclass overrides search:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [45]:
# Let's init it:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [46]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

# Vector Search with sqlitesearch
In the previous section we used minsearch for vector search.

It works, but it has three problems:
* It rebuilds the index on every startup
* It keeps everything in memory
* It searches by brute force

NN (exact):    compare query against ALL documents -> top 5 <br>
ANN (approx):  narrow down to a region -> compare within region -> top 5


sqlitesearch <br>
sqlitesearch is the persistent sibling of minsearch, and it solves both problems at once. <br>
It stores vectors in SQLite, a real on-disk database, and uses ANN strategies for retrieval. Because the data lives on disk, one process can write the vectors and another can read them back.

In [47]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

## Indexing the data
The index is saved to faq_vectors2.db. Unlike minsearch, this file persists on disk.

In [48]:
# Fit the index with our vectors and documents
vs_index.fit(vectors, documents)

## Searching
Search works the same way as with minsearch. We always encode the query into a vector first

In [49]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [50]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': 'c207b8614e',
  'course': 'data-engineering-zoomcamp',
 

In [52]:
# Filtering works the same way
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [55]:
# Now we can search
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=2)

In [56]:
results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO

In [59]:
# try it:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'